# Phase 1 — larger-scale training + label-free ablation

Cross-Slice Covert Channel Detection — EC22711.

**What this notebook does:**
1. Generates a 3x-larger frozen dataset (600 scenarios/SNR instead of 200 — 12,600 rows instead of 4,200, n=90 per test/calibration band instead of n=30).
2. Trains + evaluates the **clean-only** pipeline (your existing, verified methodology) at this larger scale. This is the **primary** result.
3. Trains + evaluates a **label-free** pipeline (no `y` used anywhere — not for training-data selection, not for calibration) as an **ablation**, on the same dataset.
4. Compares the two against a concrete, pre-committed tolerance rule (not a post-hoc judgment call): kept as a reported finding only if avg ROC-AUC drop ≤ 0.03 and max drop ≤ 0.07 across all 14 SNR×attacker combinations, per architecture.

**Before running:** Runtime → Change runtime type → GPU (T4 is fine).

**Branch note:** this clones `phase1-consolidated`, NOT `master`. As of when this notebook was written, `master` is still missing the AdaptiveAttacker perturbation-ceiling fix (it would silently reproduce the disproven "adaptive attacker is louder than non-adaptive" bug). Once PR #1 is merged into `master`, change `BRANCH` below to `"master"` — but verify that merge actually happened first; don't just assume it did.

In [ ]:
BRANCH = "phase1-consolidated"  # change to "master" only after confirming PR #1 is merged
REPO_URL = "https://github.com/pavinkishore7/network_covert_channel_detection.git"

SCENARIOS_PER_SNR = 600  # 3x the repo's default 200 -> n=90 per test/calibration band instead of n=30
SEED = 2026

# Tolerance rule for whether the label-free ablation gets reported as a finding
# vs. flagged as a degradation. Defined up front, before seeing the numbers.
MAX_AVG_AUC_DROP = 0.03
MAX_SINGLE_AUC_DROP = 0.07

In [ ]:
!git clone --branch {BRANCH} {REPO_URL} repo
%cd repo
!git log --oneline -3
!pip install -q tensorflow scikit-learn pandas numpy matplotlib

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs visible:", gpus)
if not gpus:
    print("WARNING: no GPU detected. Check Runtime > Change runtime type > GPU. "
          "This will still run on CPU but much slower.")

## 1. Generate the larger frozen dataset

Reuses `detector/generate_frozen_dataset.py`'s actual `build_dataset()` function (same seeding scheme: `scenario_seed` increments 0,1,2,... sequentially across SNR levels, same simulator/attacker construction) rather than re-implementing the generation logic — only the scenario count is overridden, via the module's `SCENARIOS_PER_SNR` constant, so this can't silently drift from the verified generation method.

In [ ]:
import sys, time
sys.path.insert(0, ".")

import numpy as np
import detector.generate_frozen_dataset as gfd

assert gfd.SCENARIOS_PER_SNR == 200, f"expected repo default 200, got {gfd.SCENARIOS_PER_SNR} — repo may have changed"
gfd.SCENARIOS_PER_SNR = SCENARIOS_PER_SNR

t0 = time.time()
X, y, snr = gfd.build_dataset()
print(f"Generated in {time.time() - t0:.1f}s")

n_scenarios_total = SCENARIOS_PER_SNR * len(gfd.SNR_LEVELS)
expected_rows = n_scenarios_total * 3
assert X.shape == (expected_rows, 200, 64), X.shape
assert np.array_equal(y, np.tile([0, 1, 2], n_scenarios_total))
assert np.array_equal(snr, np.repeat(gfd.SNR_LEVELS, SCENARIOS_PER_SNR * 3))

print(f"X: {X.shape}, {X.nbytes / 1e9:.2f} GB float32")
print(f"Rows per SNR level: {SCENARIOS_PER_SNR * 3} (train/valid/test split below)")

import os
os.makedirs("results", exist_ok=True)
np.save("results/dataset_X_3x.npy", X)
np.save("results/dataset_y_3x.npy", y)
np.save("results/dataset_snr_3x.npy", snr)

## 2. Generalized matched-split

The repo's `detector.evaluate_structured_dae.matched_split` hardcodes 200 scenarios/SNR and a fixed 140/30/30 split — it will raise on this larger dataset. This is a straightforward generalization to the same 70/15/15 ratio (140/200 = 0.7, 30/200 = 0.15), keeping the same triplet-ordering validation the original does. `target_masks_for_rows` is reused unmodified from the repo — it only depends on `row // 3`, which holds regardless of scenario count.

In [ ]:
from detector.evaluate_structured_dae import target_masks_for_rows

def matched_split_n(y: np.ndarray, snr: np.ndarray, scenarios_per_snr: int,
                     train_frac: float = 0.7, valid_frac: float = 0.15):
    rows_per_snr = scenarios_per_snr * 3
    n_train = int(round(scenarios_per_snr * train_frac))
    n_valid = int(round(scenarios_per_snr * valid_frac))
    train, valid, test = [], [], []
    for value in np.unique(snr):
        rows = np.flatnonzero(snr == value)
        if len(rows) != rows_per_snr or not np.array_equal(
            y[rows].reshape(-1, 3), np.tile([0, 1, 2], (scenarios_per_snr, 1))
        ):
            raise ValueError(f"SNR {value}: expected {scenarios_per_snr} ordered [clean, non-adaptive, adaptive] triplets")
        triplets = rows.reshape(scenarios_per_snr, 3)
        train.extend(triplets[:n_train].ravel())
        valid.extend(triplets[n_train:n_train + n_valid].ravel())
        test.extend(triplets[n_train + n_valid:].ravel())
    return np.asarray(train), np.asarray(valid), np.asarray(test)

train, valid, test = matched_split_n(y, snr, SCENARIOS_PER_SNR)
print(f"train={len(train)}, valid={len(valid)}, test={len(test)} "
      f"(test = {len(test) // len(np.unique(snr))} per SNR band, was 30 in the original 4200-row set)")

## 3. Metrics + the unified detector

`compute_metrics` / `detection_rate_at_fpr` mirror the repo's evaluate scripts exactly (same formulas, same use of `roc_auc_score` / `average_precision_score` / `roc_curve`). `AutoencoderDetector` is imported directly — same class that produced your verified results, unmodified here.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
from detector.autoencoder_detector import AutoencoderDetector

def compute_metrics(clean_scores, attack_scores, threshold):
    labels = np.r_[np.zeros(len(clean_scores)), np.ones(len(attack_scores))]
    scores = np.r_[clean_scores, attack_scores]
    return {
        "fpr": float((clean_scores > threshold).mean()),
        "detection_rate": float((attack_scores > threshold).mean()),
        "roc_auc": float(roc_auc_score(labels, scores)),
        "pr_auc": float(average_precision_score(labels, scores)),
    }

def detection_rate_at_fpr(clean_scores, attack_scores, target_fpr=0.05):
    labels = np.r_[np.zeros(len(clean_scores)), np.ones(len(attack_scores))]
    scores = np.r_[clean_scores, attack_scores]
    fpr, tpr, _ = roc_curve(labels, scores)
    return float(np.interp(target_fpr, fpr, tpr))

## 4. Pipeline runner

One function, two modes. `label_free=False` reproduces your existing verified methodology exactly (train on `y==0` rows only, calibrate thresholds from clean-only calibration scores). `label_free=True` never touches `y`: trains on every calibration/train row regardless of class, and calibrates each SNR band's threshold from the 95th percentile of **all** calibration scores at that band (not just the clean ones) — the label-free equivalent of the same idea, assuming the calibration window is mostly-clean rather than *known*-clean.

`mode="mean"` is hardcoded for scoring, matching what your merged evaluate scripts actually do (not the structured-DAE preset's own default of `"topk"`) — verified against the repo diff, not assumed.

In [ ]:
import pandas as pd

def run_pipeline(preset: str, label_free: bool, epochs: int, batch_size: int) -> pd.DataFrame:
    assert preset in ("cnn", "structured_dae")
    build = AutoencoderDetector.cnn_preset if preset == "cnn" else AutoencoderDetector.structured_dae_preset
    model = build(X.shape[1:], seed=SEED)

    train_rows = train if label_free else train[y[train] == 0]
    fit_kwargs = {"epochs": epochs, "batch_size": batch_size, "verbose": 2}
    if preset == "structured_dae":
        fit_kwargs["mask_probability"] = 0.08
    model.fit(X[train_rows], **fit_kwargs)

    calib = np.concatenate([train, valid])
    calib_y, calib_snr = y[calib], snr[calib]
    calib_masks = target_masks_for_rows(calib)
    calib_scores = model.reconstruction_error(X[calib], mode="mean", region_mask=calib_masks)

    thresholds = {}
    for snr_value in np.unique(calib_snr):
        band = calib_snr == snr_value
        pool = calib_scores[band] if label_free else calib_scores[band & (calib_y == 0)]
        thresholds[snr_value] = float(np.percentile(pool, 95))

    test_masks = target_masks_for_rows(test)
    scores = model.reconstruction_error(X[test], mode="mean", region_mask=test_masks)
    test_y, test_snr = y[test], snr[test]

    rows = []
    for snr_value in np.unique(test_snr):
        select = test_snr == snr_value
        clean = scores[select & (test_y == 0)]
        threshold = thresholds[snr_value]
        for label, name in ((1, "non_adaptive"), (2, "adaptive")):
            attack_scores = scores[select & (test_y == label)]
            row = {
                "snr": snr_value, "attack": name, "threshold": threshold,
                "n_flagged": int((attack_scores > threshold).sum()), "n_total": int(len(attack_scores)),
                "n_fp": int((clean > threshold).sum()), "n_clean": int(len(clean)),
                **compute_metrics(clean, attack_scores, threshold),
                "detection_rate_at_5pct_fpr": detection_rate_at_fpr(clean, attack_scores, target_fpr=0.05),
            }
            rows.append(row)
    return pd.DataFrame(rows)

## 5. Run all four (2 presets × clean-only/label-free)

Epoch/batch-size defaults match the repo (`cnn`: 20/8, `structured_dae`: 30/16) — kept as-is for methodological consistency with the verified baseline rather than tuned for GPU speed.

In [ ]:
runs = {}
configs = [
    ("cnn", False, 20, 8),
    ("cnn", True, 20, 8),
    ("structured_dae", False, 30, 16),
    ("structured_dae", True, 30, 16),
]
for preset, label_free, epochs, batch_size in configs:
    key = f"{preset}_{'labelfree' if label_free else 'cleanonly'}"
    print(f"=== {key} ===")
    t0 = time.time()
    df = run_pipeline(preset, label_free, epochs, batch_size)
    print(f"  done in {time.time() - t0:.1f}s")
    runs[key] = df
    df.to_csv(f"results/{key}_results_3x.csv", index=False)

print("\nAll four runs saved to results/*_results_3x.csv")

## 6. Comparison against the pre-committed tolerance rule

Kept as a reported finding only if avg ROC-AUC drop ≤ 0.03 and max drop ≤ 0.07 across all 14 SNR×attacker rows, per architecture. This is computed, not eyeballed.

In [ ]:
for preset in ("cnn", "structured_dae"):
    clean_df = runs[f"{preset}_cleanonly"].set_index(["snr", "attack"])
    lf_df = runs[f"{preset}_labelfree"].set_index(["snr", "attack"])
    delta = (clean_df["roc_auc"] - lf_df["roc_auc"]).rename("auc_drop")
    comparison = pd.concat([clean_df["roc_auc"].rename("clean_only_auc"),
                             lf_df["roc_auc"].rename("label_free_auc"), delta], axis=1)
    comparison.to_csv(f"results/{preset}_labelfree_vs_cleanonly_3x.csv")
    print(f"\n=== {preset} ===")
    print(comparison.to_string())
    avg_drop, max_drop = delta.mean(), delta.max()
    print(f"\navg AUC drop: {avg_drop:.4f} (limit {MAX_AVG_AUC_DROP})")
    print(f"max AUC drop: {max_drop:.4f} (limit {MAX_SINGLE_AUC_DROP})")
    keep = avg_drop <= MAX_AVG_AUC_DROP and max_drop <= MAX_SINGLE_AUC_DROP
    print(f"VERDICT: {'KEEP label-free as a reported finding' if keep else 'DO NOT present label-free as a finding — report clean-only only, note this ablation as a documented limitation/future-work item instead'}")

## 7. Plots

Clean-only (primary, solid) vs label-free (ablation, dashed) ROC-AUC across SNR, per architecture. Generated regardless of the verdict above — useful for your own judgment even if the tolerance rule says not to present label-free as a finding.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, preset in zip(axes, ("cnn", "structured_dae")):
    for attack, style in (("non_adaptive", "-"), ("adaptive", "--")):
        for mode, marker, alpha in (("cleanonly", "o", 1.0), ("labelfree", "x", 0.6)):
            df = runs[f"{preset}_{mode}"]
            sub = df[df["attack"] == attack].sort_values("snr")
            ax.plot(sub["snr"], sub["roc_auc"], style, marker=marker, alpha=alpha,
                    label=f"{attack} ({mode})")
    ax.set_title(f"{preset} — 3x dataset (n=90/band)")
    ax.set_xlabel("SNR (dB)")
    ax.axhline(0.5, color="gray", linewidth=0.7, linestyle=":")
axes[0].set_ylabel("ROC-AUC")
axes[0].legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig("results/labelfree_vs_cleanonly_roc_auc_3x.png", dpi=150)
plt.show()

## 8. Next steps

Everything above is written to `results/` in this Colab session (ephemeral — download before the runtime recycles, or mount Drive and change the `results/` path if you want it to persist automatically). Files produced:

- `dataset_X_3x.npy`, `dataset_y_3x.npy`, `dataset_snr_3x.npy` — the 12,600-row frozen dataset
- `{cnn,structured_dae}_{cleanonly,labelfree}_results_3x.csv` — the four full per-SNR result tables
- `{cnn,structured_dae}_labelfree_vs_cleanonly_3x.csv` — the side-by-side comparison with computed AUC drop
- `labelfree_vs_cleanonly_roc_auc_3x.png`

To bring the clean-only 3x results into the repo as the new primary numbers (if you decide the larger n is worth adopting over the existing 4,200-row results): download these CSVs, replace `results/cnn_autoencoder_results.csv` / `results/structured_dae_results.csv` in a new branch off `phase1-consolidated`, and re-run `pytest tests/` before committing — same discipline as the consolidation PR. I'd write that as another Claude Code prompt once you've actually looked at the verdict in section 6, not before — no point drafting instructions for numbers that don't exist yet.